# 06 - Offline DQN with TextEmbedder

Same FrozenLake offline DQN loop as `02_train_offline_dqn.ipynb`, but each step is rendered with `TextEmbedder`: `action` is `type: "token"` (integer id → one `embed_tokens` row), while observation/reward/episode_done/task_done are `type: "text"` (value → string via `format` → tokenizer). Whole-step template `"<action={action},{observation},{reward},{episode_done},{task_done}>"`. Reward `0.0` and done codes `0` use `skip` so those text fragments are omitted (commas stay). Heads read the last token of each step (no learnable scratch tokens).

`type: "image"` is also supported when `pretrained` is a vision-language checkpoint (not used in this notebook).

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in `09_inference.ipynb`.


In [ ]:
import torch

from mouse_core.data import (
    DataLoader,
    Augmenter,
    Selector,
    TextTokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import DqnObjective
from mouse_core.models import Model, preferred_dtype, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import TextEmbedder
from mouse_core.models.heads import DiscreteActionValueHead


DATASET_ID = "mouse-example-dataset"          # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-offline-dqn-text"   # Hugging Face model repo for push_model_to_hub
MAX_ACTIONS = 4                               # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
SEQUENCE_LENGTH = 512                         # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                # sequences per optimizer step
NUM_CYCLES = 2                               # outer train cycles (print cadence)
TRAIN_STEPS = 50                             # optimizer updates per cycle (passed to run_train)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split='train', force_download=True)

## Data pipeline

`DataLoader` samples contiguous windows up to `sequence_length` (a max) from one or more datastores. Each sequence may be shorter than the max depending on where the window starts in the store.

Pipeline order: `augmenter → selector → tokenizer → pack → embedder`.

| Stage | Role |
| --- | --- |
| **Augmenter** | `dict → dict` (`fields=` value transforms; `seed_field=` for shared draws within a `reseed` generation). Action permute sets `input_vector_field` / `output_vector_field` on `info_q_star` so Q* stays aligned. |
| **Selector** | `dict → dict` (`fields=` `input_field`/`output_field` keep/rename) |
| **Tokenizer** | `dict → StepTokens` (`input_field` / `output_field`; `objective_fields=` is `action` / `reward` / `episode_done` / `task_done`; `grouping_field=`) |

Compose `train_transform = compose(augmenter, selector, tokenizer)`.
`DataLoader(transform=train_transform)` maps each step and packs into a `TokenBatch`.
Live inference in `09_inference.ipynb` uses the tokenizer without the augmenter so chosen actions match the env.


In [ ]:
# Pipeline order: augmenter → selector → tokenizer

PRETRAINED = "Qwen/Qwen3-0.6B"
STEP_FORMAT = "<action={action},{observation},{reward},{episode_done},{task_done}>"

augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "input_vector_field": "info_q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "type": "discrete",
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

selector = Selector(
    fields=[
        {
            "input_field": "action",
        },
        {
            "input_field": "observation",
        },
        {
            "input_field": "reward",
        },
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "task_done",
        },
        {
            "input_field": "task_index",
        },
    ],
)

tokenizer = TextTokenizer(
    input_fields=[
        {
            "type": "token",
            "input_field": "action",
        },
        {
            "type": "text",
            "input_field": "observation",
            "format": "observation={observation}",
        },
        {
            "type": "text",
            "input_field": "reward",
            "format": "reward={reward}",
            "skip": 0.0,
        },
        {
            "type": "text",
            "input_field": "episode_done",
            "format": "episode_done={episode_done}",
            "skip": 0,
        },
        {
            "type": "text",
            "input_field": "task_done",
            "format": "task_done={task_done}",
            "skip": 0,
        },
    ],
    format=STEP_FORMAT,
    pretrained=PRETRAINED,
    objective_fields=[
        {
            "input_field": "action",
        },
        {
            "input_field": "reward",
        },
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "task_done",
        },
    ],
    grouping_field="task_index",
)

train_transform = compose(augmenter, selector, tokenizer)

loader = DataLoader(
    stores=stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)


## Build The Model

A Mouse Core `Model` has three main pieces:

- `TextEmbedder` maps a tokenized `TokenBatch` into pretrained embedding vectors.
- `Qwen3Backbone` processes those tokens with a transformer backbone.
- `DiscreteActionValueHead` predicts one value per discrete action.

`TextEmbedder` modality kinds:

- `text` — turn a value into a string with per-field `format` (e.g. `16` → `"16"`), put it in the step `format`, tokenize
- `token` — integer id selects `embed_tokens[id]` directly (e.g. `16` → the 16th embedding; exactly one token). No per-field `format`.
- `image` — `{field}` inserts a vision span (requires a vision-language checkpoint)

There is no learnable scratch modality on `TextEmbedder` (use `NumericEmbedder` for that). Heads read the last token of each step via `prediction_indices`.

Step template (order is left-to-right in this string):

```text
format="<action={action},{observation},{reward},{episode_done},{task_done}>"
```

Optional tokenizer `skip` omits that field's fragment/span; step-format literals such as commas are kept. Objective columns come from tokenizer `objective_fields=` (include `action`, `reward`, `episode_done`, `task_done`). Steps pack into a flat `[B, L, D]` sequence.

The backbone exposes `hidden_dim`, and the embedder and head use that same value so the pieces connect cleanly.

### What packed steps look like

Three steps (`action` = `token`, others = `text`; reward / episode_done / task_done skipped when zero):

```text
step 0:  action=0, observation=1, reward=0.0, episode_done=0, task_done=0
step 1:  action=2, observation=5, reward=0.0, episode_done=0, task_done=0
step 2:  action=1, observation=7, reward=1.0, episode_done=1, task_done=0
```

Each step fills `format="<action={action},{observation},{reward},{episode_done},{task_done}>"`.
`{action}` is one vocab embedding; the rest is one tokenized string.
`*` marks the last token of the step — the head reads it to score the **next** action.

**Step 0** → `"<action="` + `embed[0]` + `",observation=1,,,>"`

```text
<action=⟦embed 0⟧,observation=1,,,>*
```

**Step 1** → `"<action="` + `embed[2]` + `",observation=5,,,>"`

```text
<action=⟦embed 2⟧,observation=5,,,>*
```

**Step 2** → `"<action="` + `embed[1]` + `",observation=7,reward=1.0,episode_done=1,>"`

```text
<action=⟦embed 1⟧,observation=7,reward=1.0,episode_done=1,>*
```

Packed in order: step0 · step1 · step2. `prediction_indices` points at each `*`.


In [ ]:
backbone = Qwen3Backbone(pretrained=PRETRAINED)

encoder = TextEmbedder(
    hidden_dim=backbone.hidden_dim,
    pretrained=PRETRAINED,
    format=STEP_FORMAT,
    modalities=[
        {
            "type": "token",
            "field": "action",
        },
        {
            "type": "text",
            "field": "observation",
            "format": "observation={observation}",
        },
        {
            "type": "text",
            "field": "reward",
            "format": "reward={reward}",
        },
        {
            "type": "text",
            "field": "episode_done",
            "format": "episode_done={episode_done}",
        },
        {
            "type": "text",
            "field": "task_done",
            "format": "task_done={task_done}",
        },
        # Image: put {pixels} in format when pretrained is a vision-language checkpoint.
        # {"type": "image", "field": "pixels"},
    ],
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device=device, dtype=preferred_dtype(device))
print(model)


## Training Phase

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`. Mouse Core abstractions do most of the work:

1. `loader.next_batch()` samples ragged step windows (up to `SEQUENCE_LENGTH`).
2. `model(batch)` embeds the `TokenBatch`, runs the backbone with per-sequence causal attention/RoPE, and produces flat per-step head predictions.
3. `objective(objective_data, predictions)` computes the DQN loss and metrics.
4. The optimizer updates model weights.
5. `model.polyak_update(...)` updates target-network weights used by the objective.

`DqnObjective` interprets `episode_done` and `task_done` (each `0`/`1`/`2`) through separate discount factors. The bootstrap is multiplied by the episode gamma, then by the task gamma (`1.0` when `task_done` is `0`). When a task ends both fire, so a task gamma of `0.0` zeros the whole term.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-05, weight_decay=0.0, betas=(0.9, 0.95), eps=1e-08)
objective = DqnObjective(gamma_step=1.0, gamma_episode_terminal=1.0, gamma_episode_truncated=1.0, gamma_task_terminal=0.0, gamma_task_truncated=0.0, tau=0.0005, grouping_field="task_index")

def run_train(*, model: Model, optimizer: torch.optim.Optimizer, objective: DqnObjective, loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        batch = loader.next_batch()
        predictions, objective_data, _ = model(batch)
        loss, metrics = objective(objective_data, predictions)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.polyak_update(action_value_tau=objective.tau)
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the checkpoint later in `09_inference.ipynb`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(model=model, optimizer=optimizer, objective=objective, loader=loader, num_steps=TRAIN_STEPS)
    print(f"cycle={cycle} train  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}")
loader.close()


## Push To The Hub

`push_model_to_hub` saves the model architecture and weights together. Later, `load_model` can reconstruct the full `Model` without repeating the embedder, backbone, and head definitions.


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model=model, repo_id=MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")